# Transformer LM
transformer_lm.py

*   element-wise multiplies:
*   dot product:
    * init: `0 matrix multiplies` `0 FLOPs`
    * forward:
      * `num_layers * 9 matrix multiplies + 1`
      * `FLOPs = (num_layers * (((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model)))) + (batch...) * (2 * seq_len * d_model * vocab_size) FLOPs`

In [ ]:
import torch.nn as nn
from jaxtyping import Float, Int
from torch import Tensor
from cs336_basics.embedding import Embedding
from cs336_basics.transformer_block import TransformerBlock
from cs336_basics.rmsnorm_einx import RMSNorm
from cs336_basics.linear_module import Linear

class TransformerLM(nn.Module):
    def __init__(self, vocab_size: int, context_length: int, num_layers: int, d_model: int, num_heads: int, d_ff: int, rope_theta: float):
        """Implement the Transformer language model

            Args:
                vocab_size (int): The size of the vocabulary, necessary for determining the dimensionality of the token embedding matrix
                context_length (int): The maximum context length, necessary for determining the dimensionality of the position embedding matrix
                num_layers (int): The number of Transformer blocks to use
                d_model (int): Dimensionality of the Transformer block inputs
                num_heads (int): Number of heads to use in multi-head self-attention
                d_ff (int): Dimensionality of the position-wise feed-forward inner layer
                rope_theta (float): RoPE parameter
        """
        super().__init__()

        self.embedding_layer = Embedding(vocab_size, d_model) # 0 matrix multiplies; 0 FLOPs
        self.transformerblock_layer = nn.ModuleList(
            [TransformerBlock(d_model, num_heads, d_ff, context_length, rope_theta) for _ in range(num_layers)]
        ) # num_layers * (0 matrix multiplies; 0 FLOPs)
        # self.transformerblock_layer = [TransformerBlock(d_model, num_heads, d_ff, context_length, rope_theta) for _ in range(num_layers)]
        self.rmsnorm_layer = RMSNorm(d_model) # 0 matrix multiplies; 0 FLOPs
        self.linear_layer = Linear(d_model, vocab_size) # 0 matrix multiplies; 0 FLOPs
    def forward(self, in_indices: Int[Tensor, " batch_size sequence_length"]) -> Float[Tensor, " batch_size sequence_length vocab_size"]:
        """Implement the Transformer language model

            Args:
                in_indices (Int[Tensor, " batch_size sequence_length"]): Tensor with input indices to run the language model on. Shape is (batch_size, sequence_length), where
            `sequence_length` is at most `context_length`

            Returns:
                Float[Tensor, "batch_size sequence_length vocab_size"]: Tensor with the predicted unnormalized next-word distribution for each token.
        """
        input_embedding = self.embedding_layer.forward(in_indices) # 0 matrix multiplies; 0 FLOPs

        for block in self.transformerblock_layer:                  # num_layers * (9 matrix multiplies; ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model) FLOPs)
            output_embedding = block.forward(input_embedding)
            input_embedding = output_embedding

        output_embedding = self.rmsnorm_layer.forward(output_embedding) # 0 matrix multiplies; 0 FLOPs

        result = self.linear_layer.forward(output_embedding) # 1 matrix multiplies; FLOPs = (batch...) * (2 * seq_len * d_model * vocab_size) FLOPs

        return result

## TokenEmbedding

embedding.py

`0 matrix multiplies`

`0 FLOPs`

In [ ]:
import torch
import torch.nn as nn

class Embedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        """Construct an embedding module.

        Args:
            num_embeddings(int): Size of the vocabulary
            embedding_dim(int): Dimension of the embedding vectors
            device(torch.device | None): Device to store the parameters on
            dtype(torch.dtype | None): Data type of the parameters
        """
        super().__init__()
        temp_W = torch.empty(num_embeddings, embedding_dim, device=device, dtype=dtype)
        temp_W = torch.nn.init.trunc_normal_(temp_W, mean=0, std=1, a=-3, b=3)
        self.W = nn.Parameter(temp_W)
    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        """Lookup the embedding vectors for the given token IDs.

        Args:
            token_ids(torch.Tensor): token ids with shape (batch_size, sequence_length)
        """
        return self.W[token_ids]

## Transformer Block
transformer_block.py

dot product: `9 matrix multiplies`;`((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model) FLOPs`

*   external logic inside the block: `0 matrix multiplies` `0 FLOPs`
*   internal logic inside the block:
    *   RMS Norm 1: `0 matrix multiplies` `0 FLOPs`
    *   Causal Multi-Head Self-Attention w/ RoPE: `6 matrix multiplies`; `((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) FLOPs`
    *   RMS Norm 2: `0 matrix multiplies` `0 FLOPs`
    *   Position-Wise Feed-Forward: `3 matrix multiplies`; `(batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model) FLOPs`

---
* init: `0 matrix multiplies`; `0 FLOPs`
* forward: `9 matrix multiplies`;`((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model) FLOPs`









In [ ]:
import torch
import torch.nn as nn
from jaxtyping import Float
from torch import Tensor
from cs336_basics.rmsnorm_einx import RMSNorm
from cs336_basics.multihead_self_attention_rope import MultiHeadSelfAttentionRope
from cs336_basics.positionwise_feedforward_einx import PWFFN

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, max_seq_len: int, theta: float):
        """Implement the pre-norm Transformer block

            Args:
                d_model (int): Dimensionality of the Transformer block inputs
                num_heads (int): Number of heads to use in multi-head self-attention
                d_ff (int): Dimensionality of the position-wise feed-forward inner layer
                max_seq_len (int): Maximum sequence length to pre-cache if your implementation does that.
                theta (float): RoPE parameter
        """
        super().__init__()

        self.rmsnorm_first_layer = RMSNorm(d_model) # 0 matrix multiplies; 0 FLOPs
        self.rmsnorm_second_layer = RMSNorm(d_model) # 0 matrix multiplies; 0 FLOPs
        self.multiheadselfattentionrops_layer = MultiHeadSelfAttentionRope(d_model, num_heads, max_seq_len, theta) # 0 matrix multiplies, 0 FLOPs
        self.positionwiseffn_layer = PWFFN(d_model, d_ff) # 0 matrix multiplies; 0 FLOPs
    def forward(self, x: Float[Tensor, " batch sequence_length d_model"]) -> Float[Tensor, " batch sequence_length d_model"]:
        x_norm = self.rmsnorm_first_layer.forward(x) # 0 matrix multiplies; 0 FLOPs
        # embedding_attention = self.multiheadselfattentionrops_layer.forward(x_norm)
        x_seq_len = x.size(-2)
        token_positions = torch.arange(x_seq_len).unsqueeze(0).expand(*x.shape[:-2], x_seq_len)
        embedding_attention = self.multiheadselfattentionrops_layer.forward(x_norm, token_positions) # 6 matrix multiplies; FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) FLOPs

        result_firstsublayer = x + embedding_attention

        result_firstsublayer_norm = self.rmsnorm_second_layer.forward(result_firstsublayer) # 0 matrix multiplies; 0 FLOPs
        embedding_pwffn = self.positionwiseffn_layer.forward(result_firstsublayer_norm) # 3 matrix multiplies; (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model) FLOPs
        result_secondsublayer = result_firstsublayer + embedding_pwffn

        return result_secondsublayer

### Norm
rmsnorm_einx.py

*   element-wise multiplies:
*   dot product: `0 matrix multiplies` `0 FLOPs`


In [ ]:
import torch
import torch.nn as nn
import einx

class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5, device: torch.device | None = None, dtype: torch.dtype | None = None):
        """ Construct the RMSNorm module.

        Args:
            d_model(int): Hidden dimension of the model
            eps(float): Epsilon value for numerical stability
            device(torch.device | None = None): Device to store the parameters on
            dtype(torch.dtype | None = None): Data type of the parameters
        """
        super().__init__()
        self.g = nn.Parameter(torch.ones(d_model, dtype=dtype, device=device))
        self.eps = eps
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Process an input tensor of shape

        Args:
            x (torch.Tensor): shape (batch_size, sequence_length, d_model)
        Returns:
            torch.Tensor: shape (batch_size, sequence_length, d_model)
        """
        in_dtype = x.dtype
        x = x.to(torch.float32)

        mean_square = einx.mean('... d -> ... 1', x * x)
        inv_rms = torch.rsqrt(mean_square + self.eps)

        x_norm = einx.multiply('... d, ... 1 -> ... d', x, inv_rms)
        result = einx.multiply('... d, d -> ... d', x_norm, self.g)

        return result.to(in_dtype)


### Causal Multi-Head Self-Attention w/ RoPE
multihead_self_attention_rope.py

*   element-wise multiplies:
*   dot product: `6 matrix multiplies`; `((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) FLOPs`
    * init: `0 matrix multiplies`, `0 FLOPs`
    * forward:
      * `6 matrix multiplies`
      * `FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) FLOPs`

In [ ]:
import torch
import torch.nn as nn
from jaxtyping import Float, Int
from torch import Tensor
from cs336_basics.scaled_dot_product_attention import SDPAttention
from cs336_basics.rope_einx import RoPe

class MultiHeadSelfAttentionRope(nn.Module):
    def __init__(self, d_model: int, num_heads: int, max_seq_len: int, theta: float):
        """Causal multi-head self-attention

            Args:
                d_model (int): Dimensionality of the Transformer block inputs
                num_heads (int): Number of heads to use in multi-head self-attention
                max_seq_len (int): Maximum sequence length to pre-cache
                theta (float): RoPE parameter
        """
        super().__init__()

        self.Q = nn.Parameter(torch.randn(d_model, d_model))
        self.K = nn.Parameter(torch.randn(d_model, d_model))
        self.V = nn.Parameter(torch.randn(d_model, d_model))
        self.O = nn.Parameter(torch.randn(d_model, d_model))

        self.num_heads = num_heads

        self.rope_layer = RoPe(theta=theta, d_k=d_model // num_heads, max_seq_len=max_seq_len) # 0 matrix multiplies, 0 FLOPs

    def forward(self, x: Float[Tensor, " ... sequence_length d_in"], token_positions: Int[Tensor, " ... sequence_length"] | None = None) -> Float[Tensor, " ... sequence_length d_out"]:
        batch_shape = x.shape[:-2]
        seq_len = x.shape[-2]

        q_x = x @ self.Q.T  # x(... sequence_length d_model), self.Q.T(d_model, d_model); FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model)
        k_x = x @ self.K.T  # x(... sequence_length d_model), self.K.T(d_model, d_model); FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model)
        v_x = x @ self.V.T  # x(... sequence_length d_model), self.V.T(d_model, d_model); FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model)

        q_x_heads = q_x.reshape(*batch_shape, seq_len, self.num_heads, -1)
        k_x_heads = k_x.reshape(*batch_shape, seq_len, self.num_heads, -1)
        v_x_heads = v_x.reshape(*batch_shape, seq_len, self.num_heads, -1)

        q_x_heads = q_x_heads.transpose(-3, 2)
        k_x_heads = k_x_heads.transpose(-3, 2)
        v_x_heads = v_x_heads.transpose(-3, 2)

        q_x_heads = self.rope_layer.forward(q_x_heads, token_positions) # 0 matrix multiplies, 0 FLOPs
        k_x_heads = self.rope_layer.forward(k_x_heads, token_positions) # 0 matrix multiplies, 0 FLOPs

        causal_mask = ~torch.triu(torch.ones(x.shape[-2], x.shape[-2], dtype=bool), diagonal=1)

        attention_layer = SDPAttention(q_x_heads, k_x_heads, v_x_heads, causal_mask) # 0 matrix multiplies; 0 FLOPs
        embedding_cmhsa = attention_layer.forward() # 2 matrix multiplies; (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) FLOPs
        embedding_cmhsa_trans = embedding_cmhsa.transpose(-3, -2)
        embedding_cmhsa_combined = embedding_cmhsa_trans.contiguous().reshape(*batch_shape, seq_len, -1)
        result = embedding_cmhsa_combined @ self.O.T # embedding_cmhsa_combined(batch..., seq_len, d_model), self.O.T(d_model, d_model); FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model)

        return result

#### Scaled Dot-Product Attention
scaled_dot_product_attention.py

*   element-wise multiplies:
*   dot product:
    * init: `0 matrix multiplies`, `0 FLOPs`
    * forward:
      * `2 matrix multiplies`
      * `(batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) FLOPs`

In [ ]:
import torch
import torch.nn as nn
from jaxtyping import Float
from torch import Tensor
import cs336_basics.softmax_einx as sm
import math

class SDPAttention(nn.Module):
    def __init__(self, q: Float[Tensor, "... seq_len d_k"], k: Float[Tensor, "... seq_len d_k"], v: Float[Tensor, "... seq_len d_v"], mask: Float[Tensor, " ... queries keys"] | None = None):
        super().__init__()

        self.Q = q
        self.K = k
        self.V = v
        self.mask = mask
    def forward(self) -> Float[Tensor, "... d_v"]:
        qk = self.Q @ self.K.transpose(-2, -1) # self.Q(... seq_len d_model), self.K.transpose(-2, -1)(... d_model seq_len); FLOPs = (batch...) * (2 * seq_len * d_model * seq_len)
        qk_norm = qk / math.sqrt(self.Q.size(-1))

        mask_ninf = torch.where(self.mask, torch.zeros_like(self.mask), float('-inf'))

        qk_norm_mask = qk_norm + mask_ninf

        attention_score = sm.Softmax(qk_norm_mask, -1) # 0 matrix multiplies, 0 FLOPs
        result = attention_score @ self.V # attention_score(... seq_len seq_len), self.V(... seq_len d_model); FLOPs = (batch...) * (2 * seq_len * seq_len * d_model)

        return result



##### Softmax
softmax_einx.py

*   element-wise multiplies:
*   dot product: `0 matrix multiplies`, `0 FLOPs`

In [ ]:
import torch
from jaxtyping import Float
from torch import Tensor
import einx

def Softmax(x: Float[Tensor, " ..."], dim: int) -> Float[Tensor, " ..."]:
    logit_stable = einx.subtract("... logits, ... 1 -> ... logits", x, x.max(dim=dim, keepdim=True).values)
    logit_stable_exp = torch.exp(logit_stable)
    result = einx.divide("... logits, ... 1 -> ... logits", logit_stable_exp, logit_stable_exp.sum(dim=dim, keepdim=True))

    return result


#### RoPE
rope_einx.py

*   element-wise multiplies:
*   dot product: `0 matrix multiplies` `0 FLOPs`
    * init: `0 matrix multiplies` `0 FLOPs`
    * forward: `0 matrix multiplies` `0 FLOPs`

In [ ]:
import torch
import torch.nn as nn
from jaxtyping import Float, Int
from torch import Tensor
import einx

class RoPe(nn.Module):
    def __init__(self, theta: float, d_k: int, max_seq_len: int, device: torch.device | None = None):
        """Constructthe RoPE module and create buffers if needed.

        Args:
            theta (float): Θ value for the RoPE
            d_k (int): dimension of query and key vectors
            max_seq_len (int): Maximum sequence length that will be inputted
            device (torch.device | None): Device to store the buffer on
        """
        super().__init__()

        block_num = d_k // 2

        angle_i = torch.arange(max_seq_len)
        angle_k = torch.arange(1, block_num + 1)
        angle = einx.multiply("max_seq_len 1, 1 block_num -> max_seq_len block_num", angle_i[:, None], torch.reciprocal(theta ** ((2*angle_k[None, :] - 2) / d_k)))

        sin = torch.sin(angle)
        cos = torch.cos(angle)

        self.register_buffer("sin", sin, persistent=False)
        self.register_buffer("cos", cos, persistent=False)

    def forward(self, x: Float[Tensor, "... seq_len d_k"], token_positions: Int[Tensor, "... seq_len"]) -> Float[Tensor, "... seq_len d_k"]:
        *batch, seq_len, d_k = x.shape
        block = d_k // 2

        x_blocked = x.reshape(*batch, seq_len, block, -1)

        x_even = x_blocked[..., 0]
        x_odd = x_blocked[..., 1]

        sin_pos = self.sin[token_positions]
        cos_pos = self.cos[token_positions]

        x_even_rot = x_even * cos_pos - x_odd * sin_pos
        x_odd_rot = x_even * sin_pos + x_odd * cos_pos

        result_blocked = torch.stack((x_even_rot, x_odd_rot), dim=-1)
        result = result_blocked.reshape(*batch, seq_len, -1)

        return result


### Norm
rmsnorm_einx.py
*   element-wise multiplies:
*   dot product: `0 matrix multiplies` `0 FLOPs`

In [ ]:
import torch
import torch.nn as nn
import einx

class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5, device: torch.device | None = None, dtype: torch.dtype | None = None):
        """ Construct the RMSNorm module.

        Args:
            d_model(int): Hidden dimension of the model
            eps(float): Epsilon value for numerical stability
            device(torch.device | None = None): Device to store the parameters on
            dtype(torch.dtype | None = None): Data type of the parameters
        """
        super().__init__()
        self.g = nn.Parameter(torch.ones(d_model, dtype=dtype, device=device))
        self.eps = eps
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Process an input tensor of shape

        Args:
            x (torch.Tensor): shape (batch_size, sequence_length, d_model)
        Returns:
            torch.Tensor: shape (batch_size, sequence_length, d_model)
        """
        in_dtype = x.dtype
        x = x.to(torch.float32)

        mean_square = einx.mean('... d -> ... 1', x * x)
        inv_rms = torch.rsqrt(mean_square + self.eps)

        x_norm = einx.multiply('... d, ... 1 -> ... d', x, inv_rms)
        result = einx.multiply('... d, d -> ... d', x_norm, self.g)

        return result.to(in_dtype)


### Position-Wise Feed-Forward
positionwise_feedforward_einx.py

*   element-wise multiplies:
*   dot product: `3 matrix multiplies`; `(batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model) FLOPs`
    * init: `0 matrix multiplies`, `0 FLOPs`
    * forward:
      * `3 matrix multiplies`
      * `(batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model) FLOPs`

In [ ]:
import torch
import torch.nn as nn
import einx
from jaxtyping import Float
from torch import Tensor

class PWFFN(nn.Module):
    def __init__(self, d_model: int, d_ff: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()

        # d_ff = 8/3 * d_model

        self.W1 = nn.Parameter(torch.randn(d_ff, d_model, dtype=dtype, device=device))
        self.W3 = nn.Parameter(torch.randn(d_ff, d_model, dtype=dtype, device=device))

        self.W2 = nn.Parameter(torch.randn(d_model, d_ff, dtype=dtype, device=device))
    def forward(self, x: Float[Tensor, " ... d_model"]) -> Float[Tensor, "... d_model"]:
        w1_item = einx.dot("... [d_model], [d_model] d_ff -> ... d_ff", x, self.W1.T) # x(... [d_model]), self.W1.T([d_model] d_ff); FLOPs = (batch...) * (2 * 1 * d_model * d_ff)
        w1_gate_item = PWFFN.silu(w1_item)

        w3_item = einx.dot("... [d_model], [d_model] d_ff -> ... d_ff", x, self.W3.T) # x(... [d_model]), self.W3.T([d_model] d_ff); FLOPs = (batch...) * (2 * 1 * d_model * d_ff)

        l1 = einx.multiply("... d_ff, ... d_ff -> ... d_ff", w1_gate_item, w3_item)
        result = einx.dot("... [d_ff], [d_ff] d_model -> ... d_model", l1, self.W2.T) # l1(... [d_ff]), self.W2.T([d_ff] d_model); FLOPs = (batch...) * (2 * 1 * d_ff * d_model)

        return result

    @staticmethod
    def silu(x: Float[Tensor, "... d"]) -> Float[Tensor, "... d"]:
        return x * torch.sigmoid(x)

## Norm
rmsnorm_einx.py

*   element-wise multiplies:
*   dot product: `0 matrix multiplies` `0 FLOPs`


In [ ]:
import torch
import torch.nn as nn
import einx

class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5, device: torch.device | None = None, dtype: torch.dtype | None = None):
        """ Construct the RMSNorm module.

        Args:
            d_model(int): Hidden dimension of the model
            eps(float): Epsilon value for numerical stability
            device(torch.device | None = None): Device to store the parameters on
            dtype(torch.dtype | None = None): Data type of the parameters
        """
        super().__init__()
        self.g = nn.Parameter(torch.ones(d_model, dtype=dtype, device=device))
        self.eps = eps
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Process an input tensor of shape

        Args:
            x (torch.Tensor): shape (batch_size, sequence_length, d_model)
        Returns:
            torch.Tensor: shape (batch_size, sequence_length, d_model)
        """
        in_dtype = x.dtype
        x = x.to(torch.float32)

        mean_square = einx.mean('... d -> ... 1', x * x)
        inv_rms = torch.rsqrt(mean_square + self.eps)

        x_norm = einx.multiply('... d, ... 1 -> ... d', x, inv_rms)
        result = einx.multiply('... d, d -> ... d', x_norm, self.g)

        return result.to(in_dtype)


## Linear(Output Embedding)
linear_module.py

*   element-wise multiplies:
*   dot product:
    * init: `0 matrix multiplies` `0 FLOPs`
    * forward:
      * `1 matrix multiplies`
      * `FLOPs = (batch...) * (2 * seq_len * d_model * vocab_size) FLOPs`

In [ ]:
import torch
import torch.nn as nn
import math

class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        """Construct a linear transformation module.

        Args:
            in_features(int): final dimension of the input
            out_features(int): final dimension of the output
            device(torch.device | None): Device to store the parameters on
            dtype(torch.dtype | None): Data type of the parameters
        """
        super().__init__()
        self.W = nn.Parameter(torch.randn(out_features, in_features, dtype=dtype, device=device))
        std_variance = math.sqrt(2/(in_features + out_features))
        nn.init.trunc_normal_(self.W, mean=0, std=std_variance, a=-3*std_variance, b=3*std_variance)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the linear transformation to the input.
        """
        return x @ self.W.T # x(batch_size, seq_len, d_model), self.W.T(d_model, vocab_size); FLOPs = (batch...) * (2 * seq_len * d_model * vocab_size)